# LeetCode 1244 — Design A Leaderboard bằng Python

Link đề: https://leetcode.com/problems/design-a-leaderboard/ · Độ khó: **Medium** · Chủ đề: Hash Table, Design, Sorting

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề, tên class và chữ ký các method được đối chiếu với https://leetcode.doocs.org/en/lc/1244/ (class `Leaderboard`, các method `addScore(playerId, score)`, `top(K)`, `reset(playerId)`; tham số `K` viết **hoa**).

Skeleton LeetCode:

```python
class Leaderboard:

    def __init__(self):
        ...

    def addScore(self, playerId: int, score: int) -> None:
        ...

    def top(self, K: int) -> int:
        ...

    def reset(self, playerId: int) -> None:
        ...


# Your Leaderboard object will be instantiated and called as such:
# obj = Leaderboard()
# obj.addScore(playerId,score)
# param_2 = obj.top(K)
# obj.reset(playerId)
```

Ghi chú: đây là bài **thiết kế** (Design) nên không có `class Solution`; LeetCode khởi tạo `Leaderboard()` rồi gọi lần lượt các method theo danh sách thao tác.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu cách LeetCode chấm bài Design: một danh sách tên thao tác + danh sách tham số, đầu ra là danh sách giá trị trả về (`None` cho method không trả gì) — và tự viết **bộ chạy giả lập** (harness) cho nó.
2. Dùng `dict` (hash table) để lưu điểm từng người chơi với `addScore`/`reset` **O(1)**.
3. Cài đặt `top(K)` theo nhiều cách: **sắp xếp** toàn bộ, **heap** (`heapq.nlargest` / min-heap kích thước K), **đếm theo điểm** (bucket), và **danh sách luôn được sắp xếp** bằng `bisect`.
4. Phân tích đánh đổi (trade-off) giữa chi phí cập nhật và chi phí truy vấn — câu hỏi cốt lõi của mọi bài thiết kế.
5. Tránh các lỗi: ghi đè điểm thay vì cộng dồn, quên xoá điểm cũ khỏi cấu trúc sắp xếp, phá hỏng trạng thái khi `pop` heap trong `top`.

## 2. Đề bài & ví dụ

Thiết kế class `Leaderboard` (bảng xếp hạng) với 3 hàm:

- `addScore(playerId, score)`: cộng `score` vào điểm của người chơi `playerId`. Nếu chưa có người chơi này trên bảng, thêm họ với số điểm `score`.
- `top(K)`: trả về **tổng điểm của `K` người chơi điểm cao nhất**.
- `reset(playerId)`: đặt lại điểm của người chơi về 0 (nói cách khác, **xoá** họ khỏi bảng). Đảm bảo người chơi đã được thêm trước khi gọi hàm này.

Ban đầu bảng rỗng.

**Ví dụ 1:**

```
Input:
["Leaderboard","addScore","addScore","addScore","addScore","addScore","top","reset","reset","addScore","top"]
[[],[1,73],[2,56],[3,39],[4,51],[5,4],[1],[1],[2],[2,51],[3]]
Output:
[null,null,null,null,null,null,73,null,null,null,141]
```

Giải thích: sau 5 lần `addScore`, bảng là `[[1,73],[2,56],[3,39],[4,51],[5,4]]`; `top(1) = 73`; `reset(1)`, `reset(2)` → `[[3,39],[4,51],[5,4]]`; `addScore(2,51)` → `[[2,51],[3,39],[4,51],[5,4]]`; `top(3) = 51 + 51 + 39 = 141`.

**Ràng buộc:**

- `1 <= playerId, K <= 10000`
- Đảm bảo `K` không vượt quá số người chơi hiện tại.
- `1 <= score <= 100`
- Tối đa `1000` lời gọi hàm.

Bộ chạy giả lập `run_ops(cls, ops, args)` làm đúng việc của trình chấm: tạo đối tượng từ thao tác đầu tiên, rồi gọi method theo tên bằng `getattr`.

In [1]:
import bisect
import heapq
import random
import time
from collections import defaultdict, Counter
from typing import List


def run_ops(cls, ops, args):
    out = []
    obj = None
    for op, a in zip(ops, args):
        if op == "Leaderboard":
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


ops1 = ["Leaderboard", "addScore", "addScore", "addScore", "addScore", "addScore", "top", "reset", "reset", "addScore", "top"]
args1 = [[], [1, 73], [2, 56], [3, 39], [4, 51], [5, 4], [1], [1], [2], [2, 51], [3]]
want1 = [None, None, None, None, None, None, 73, None, None, None, 141]
print(len(ops1), len(args1), want1)

11 11 [None, None, None, None, None, None, 73, None, None, None, 141]


## 3. Kiến thức nền

### 3.1 `dict` và `defaultdict(int)` — lưu điểm từng người

`dict` cho tra cứu/cập nhật trung bình O(1). `defaultdict(int)` tự tạo giá trị `0` cho khoá chưa có, nên `d[p] += score` dùng được ngay cả với người chơi mới. `d.pop(p)` xoá và trả về giá trị.

In [2]:
d = defaultdict(int)
d[1] += 73
d[2] += 56
d[1] += 10
print(dict(d))
print("pop(2) =", d.pop(2), "->", dict(d))

{1: 83, 2: 56}
pop(2) = 56 -> {1: 83}


### 3.2 `sorted(..., reverse=True)` và cắt lát

Cách thẳng nhất để lấy K số lớn nhất: sắp xếp giảm dần rồi lấy `[:K]`. Tốn `O(n log n)`.

In [3]:
vals = [73, 56, 39, 51, 4]
print(sorted(vals, reverse=True), "-> top 3 =", sum(sorted(vals, reverse=True)[:3]))

[73, 56, 51, 39, 4] -> top 3 = 180


### 3.3 `heapq` — hàng đợi ưu tiên

`heapq` của Python là **min-heap** (phần tử nhỏ nhất ở `h[0]`).

- `heapq.nlargest(K, iterable)`: K phần tử lớn nhất, `O(n log K)`.
- Tự làm: giữ min-heap kích thước K; gặp số lớn hơn đỉnh thì `heapreplace`. Cuối cùng heap chứa đúng K số lớn nhất.

In [4]:
print(heapq.nlargest(3, vals))
h = []
for v in vals:
    if len(h) < 3:
        heapq.heappush(h, v)
    elif v > h[0]:
        heapq.heapreplace(h, v)
print("min-heap kích thước 3:", sorted(h), "tổng", sum(h))

[73, 56, 51]
min-heap kích thước 3: [51, 56, 73] tổng 180


### 3.4 `bisect` — giữ một list luôn sắp xếp

- `bisect.insort(a, x)`: chèn `x` vào đúng chỗ, `O(log n)` để tìm + `O(n)` để dịch phần tử.
- Xoá một giá trị `x`: `i = bisect.bisect_left(a, x)` rồi `a.pop(i)` (hoặc `del a[i]`).
- Với list tăng dần, K số lớn nhất là `a[-K:]`.

In [5]:
a = []
for v in vals:
    bisect.insort(a, v)
print(a)
i = bisect.bisect_left(a, 56)
a.pop(i)
print("xoá 56:", a, "| top 2 =", sum(a[-2:]))

[4, 39, 51, 56, 73]
xoá 56: [4, 39, 51, 73] | top 2 = 124


### 3.5 `Counter` — đếm số người theo từng mức điểm

Nhiều người có thể **cùng điểm**. Thay vì lưu từng người, ta có thể lưu `cnt[điểm] = số người có điểm đó`. Khi tính top K, đi từ điểm cao xuống, lấy `min(cnt, còn lại)` người ở mỗi mức.

In [6]:
cnt = Counter([51, 51, 39, 4])
print(cnt)
K, total = 3, 0
for s in sorted(cnt, reverse=True):
    take = min(cnt[s], K)
    total += take * s
    K -= take
    if K == 0:
        break
print("top 3 =", total)

Counter({51: 2, 39: 1, 4: 1})
top 3 = 141


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1, theo dõi `dict` điểm:

| Thao tác | Bảng sau thao tác | Trả về |
|---|---|---|
| addScore(1,73) | {1:73} | null |
| addScore(2,56) | {1:73, 2:56} | null |
| addScore(3,39) | {1:73, 2:56, 3:39} | null |
| addScore(4,51) | {…, 4:51} | null |
| addScore(5,4) | {1:73, 2:56, 3:39, 4:51, 5:4} | null |
| top(1) | (không đổi) | 73 |
| reset(1) | {2:56, 3:39, 4:51, 5:4} | null |
| reset(2) | {3:39, 4:51, 5:4} | null |
| addScore(2,51) | {3:39, 4:51, 5:4, 2:51} | null |
| top(3) | (không đổi) | 51+51+39 = 141 |

Quan sát: `addScore` và `reset` chỉ đụng tới **một** người chơi; `top` cần nhìn **tất cả**. Ta phải chọn: đầu tư cấu trúc để `top` nhanh (trả giá khi cập nhật), hay giữ cập nhật O(1) và trả giá khi `top`.

In [7]:
board = {}
for op, a in zip(ops1[1:], args1[1:]):
    if op == "addScore":
        board[a[0]] = board.get(a[0], 0) + a[1]
    elif op == "reset":
        del board[a[0]]
    else:
        print(f"top({a[0]}) =", sum(sorted(board.values(), reverse=True)[:a[0]]), "| bảng:", board)

top(1) = 73 | bảng: {1: 73, 2: 56, 3: 39, 4: 51, 5: 4}
top(3) = 141 | bảng: {3: 39, 4: 51, 5: 4, 2: 51}


## 5. Cách 1 — Brute Force / baseline

`dict` lưu điểm; `top(K)` sắp xếp **toàn bộ** điểm giảm dần rồi cộng K số đầu.

In [8]:
class LeaderboardSort:
    def __init__(self):
        self.score = defaultdict(int)

    def addScore(self, playerId, score):
        self.score[playerId] += score

    def top(self, K):
        return sum(sorted(self.score.values(), reverse=True)[:K])

    def reset(self, playerId):
        self.score.pop(playerId, None)


out = run_ops(LeaderboardSort, ops1, args1)
print(out, out == want1)

[None, None, None, None, None, None, 73, None, None, None, 141] True


**Độ phức tạp:** gọi `P` là số người chơi hiện tại. `addScore`, `reset`: O(1). `top`: `O(P log P)`. Với tối đa 1000 lời gọi thì `P ≤ 1000`, nên thực ra cách này **đã đủ nhanh** cho đề — nhưng là baseline để so sánh.

**Nút thắt:** mỗi `top(K)` sắp xếp lại từ đầu, kể cả khi chỉ có một người vừa đổi điểm, và kể cả khi `K` rất nhỏ.

## 6. Bước đột phá

Hai hướng cải tiến, tuỳ thao tác nào xuất hiện nhiều hơn:

1. **Giữ cập nhật O(1), làm `top` rẻ hơn:** không cần sắp xếp hết, chỉ cần K lớn nhất → **heap** `O(P log K)`; hoặc khai thác việc điểm là **số nguyên bị chặn** → **đếm theo mức điểm** (bucket), `top` duyệt các mức điểm từ cao xuống.
2. **Giữ một cấu trúc luôn sắp xếp** (sorted list / cây cân bằng): `addScore` = xoá điểm cũ + chèn điểm mới, `reset` = xoá; `top(K)` = cộng K phần tử cuối.

**Invariant then chốt cho hướng 2:** "multiset các giá trị trong cấu trúc sắp xếp **luôn bằng đúng** multiset `score.values()`". Mọi thao tác phải giữ invariant này: khi đổi điểm người chơi, **xoá đúng một bản** điểm cũ rồi chèn điểm mới.

**Quy tắc:** nếu người chơi đã tồn tại thì *xoá điểm cũ → cộng → chèn điểm mới*; nếu chưa có thì *chèn điểm mới*; `reset` thì *xoá điểm hiện tại và xoá khỏi dict*.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `dict` + `heapq.nlargest`

In [9]:
class LeaderboardHeap:
    def __init__(self):
        self.score = defaultdict(int)

    def addScore(self, playerId, score):
        self.score[playerId] += score

    def top(self, K):
        return sum(heapq.nlargest(K, self.score.values()))

    def reset(self, playerId):
        del self.score[playerId]


out = run_ops(LeaderboardHeap, ops1, args1)
print(out, out == want1)

[None, None, None, None, None, None, 73, None, None, None, 141] True


**Độ phức tạp:** `top`: `O(P log K)` thời gian, `O(K)` bộ nhớ phụ. Tốt khi `K` nhỏ so với `P`. Cập nhật O(1).

### 7.2 Cách 3 — Min-heap kích thước K viết tay

Cùng ý tưởng với 7.1 nhưng viết tay để hiểu cơ chế: duyệt mọi điểm, giữ min-heap K phần tử; phần tử nhỏ nhất trong heap là "ngưỡng" để vào top K.

In [10]:
class LeaderboardHeapK:
    def __init__(self):
        self.score = {}

    def addScore(self, playerId, score):
        self.score[playerId] = self.score.get(playerId, 0) + score

    def top(self, K):
        h = []
        for v in self.score.values():
            if len(h) < K:
                heapq.heappush(h, v)
            elif v > h[0]:
                heapq.heapreplace(h, v)
        return sum(h)

    def reset(self, playerId):
        del self.score[playerId]


out = run_ops(LeaderboardHeapK, ops1, args1)
print(out, out == want1)

[None, None, None, None, None, None, 73, None, None, None, 141] True


### 7.3 Cách 4 — Đếm theo mức điểm (bucket) + danh sách mức điểm sắp xếp

Lưu `cnt[s]` = số người có điểm `s`, và `levels` = list **tăng dần** các mức điểm đang có người (dùng `bisect`). `top(K)` đi từ mức cao nhất xuống. Số mức điểm khác nhau `D ≤ P`, và thường nhỏ hơn nhiều khi nhiều người trùng điểm.

In [11]:
class LeaderboardBucket:
    def __init__(self):
        self.score = {}
        self.cnt = Counter()
        self.levels = []               # các mức điểm có cnt > 0, tăng dần

    def _add_level(self, s):
        if self.cnt[s] == 0:
            bisect.insort(self.levels, s)
        self.cnt[s] += 1

    def _remove_level(self, s):
        self.cnt[s] -= 1
        if self.cnt[s] == 0:
            del self.cnt[s]
            self.levels.pop(bisect.bisect_left(self.levels, s))

    def addScore(self, playerId, score):
        old = self.score.get(playerId)
        if old is not None:
            self._remove_level(old)
        new = (old or 0) + score
        self.score[playerId] = new
        self._add_level(new)

    def top(self, K):
        total = 0
        for s in reversed(self.levels):
            take = min(self.cnt[s], K)
            total += take * s
            K -= take
            if K == 0:
                break
        return total

    def reset(self, playerId):
        self._remove_level(self.score.pop(playerId))


out = run_ops(LeaderboardBucket, ops1, args1)
print(out, out == want1)

[None, None, None, None, None, None, 73, None, None, None, 141] True


**Độ phức tạp:** `addScore`/`reset`: `O(D)` worst (do `insort`/`pop` dịch phần tử trong list), thực tế rất nhanh vì `D` nhỏ và thao tác dịch là C-level. `top(K)`: `O(số mức duyệt) ≤ O(min(K, D))`.

### 7.4 Cách 5 — Sorted list toàn bộ điểm (mô phỏng `SortedList`)

Lời giải doocs dùng `sortedcontainers.SortedList` (có sẵn trên LeetCode nhưng **không** thuộc thư viện chuẩn). Ta mô phỏng bằng list + `bisect`: giữ `ranked` là list tăng dần **mọi** điểm của mọi người chơi.

In [12]:
class LeaderboardSortedList:
    def __init__(self):
        self.score = {}
        self.ranked = []

    def addScore(self, playerId, score):
        if playerId in self.score:
            old = self.score[playerId]
            self.ranked.pop(bisect.bisect_left(self.ranked, old))   # xoá 1 bản điểm cũ
            self.score[playerId] = old + score
        else:
            self.score[playerId] = score
        bisect.insort(self.ranked, self.score[playerId])

    def top(self, K):
        return sum(self.ranked[-K:])

    def reset(self, playerId):
        old = self.score.pop(playerId)
        self.ranked.pop(bisect.bisect_left(self.ranked, old))


out = run_ops(LeaderboardSortedList, ops1, args1)
print(out, out == want1)

[None, None, None, None, None, None, 73, None, None, None, 141] True


**Độ phức tạp:** `addScore`/`reset`: `O(log P)` tìm + `O(P)` dịch (với `SortedList` thật là `O(log P)` khấu hao). `top(K)`: `O(K)`. Hợp khi `top` được gọi rất nhiều.

## 8. TRACE từng bước

Trace Cách 5 (sorted list) trên ví dụ 1 — bảng trạng thái:

| Thao tác | `score` | `ranked` | Trả về |
|---|---|---|---|
| addScore(1,73) | {1:73} | [73] | None |
| addScore(2,56) | {1:73,2:56} | [56,73] | None |
| addScore(3,39) | +3:39 | [39,56,73] | None |
| addScore(4,51) | +4:51 | [39,51,56,73] | None |
| addScore(5,4) | +5:4 | [4,39,51,56,73] | None |
| top(1) | | | 73 |
| reset(1) | −1 | [4,39,51,56] | None |
| reset(2) | −2 | [4,39,51] | None |
| addScore(2,51) | +2:51 | [4,39,51,51] | None |
| top(3) | | | 39+51+51 = 141 |

In [13]:
def trace_leaderboard(ops, args):
    lb = None
    for op, a in zip(ops, args):
        if op == "Leaderboard":
            lb = LeaderboardSortedList()
            print("Leaderboard() -> rỗng")
            continue
        ret = getattr(lb, op)(*a)
        print(f"{op}{tuple(a)!s:10} -> trả về {ret!s:5} | score={lb.score} | ranked={lb.ranked}")


trace_leaderboard(ops1, args1)

Leaderboard() -> rỗng
addScore(1, 73)    -> trả về None  | score={1: 73} | ranked=[73]
addScore(2, 56)    -> trả về None  | score={1: 73, 2: 56} | ranked=[56, 73]
addScore(3, 39)    -> trả về None  | score={1: 73, 2: 56, 3: 39} | ranked=[39, 56, 73]
addScore(4, 51)    -> trả về None  | score={1: 73, 2: 56, 3: 39, 4: 51} | ranked=[39, 51, 56, 73]
addScore(5, 4)     -> trả về None  | score={1: 73, 2: 56, 3: 39, 4: 51, 5: 4} | ranked=[4, 39, 51, 56, 73]
top(1,)       -> trả về 73    | score={1: 73, 2: 56, 3: 39, 4: 51, 5: 4} | ranked=[4, 39, 51, 56, 73]
reset(1,)       -> trả về None  | score={2: 56, 3: 39, 4: 51, 5: 4} | ranked=[4, 39, 51, 56]
reset(2,)       -> trả về None  | score={3: 39, 4: 51, 5: 4} | ranked=[4, 39, 51]
addScore(2, 51)    -> trả về None  | score={3: 39, 4: 51, 5: 4, 2: 51} | ranked=[4, 39, 51, 51]
top(3,)       -> trả về 141   | score={3: 39, 4: 51, 5: 4, 2: 51} | ranked=[4, 39, 51, 51]


Trace thêm một trường hợp cộng dồn (cùng người chơi nhiều lần) với Cách 4 (bucket), để thấy `levels` và `cnt` thay đổi:

In [14]:
def trace_bucket(ops, args):
    lb = LeaderboardBucket()
    for op, a in zip(ops, args):
        ret = getattr(lb, op)(*a)
        print(f"{op}{tuple(a)!s:9} -> {ret!s:5} | levels={lb.levels} cnt={dict(lb.cnt)}")


trace_bucket(["addScore", "addScore", "addScore", "top", "addScore", "top", "reset", "top"],
             [[1, 10], [2, 10], [1, 5], [2], [2, 5], [1], [1], [1]])

addScore(1, 10)   -> None  | levels=[10] cnt={10: 1}
addScore(2, 10)   -> None  | levels=[10] cnt={10: 2}
addScore(1, 5)    -> None  | levels=[10, 15] cnt={10: 1, 15: 1}
top(2,)      -> 25    | levels=[10, 15] cnt={10: 1, 15: 1}
addScore(2, 5)    -> None  | levels=[15] cnt={15: 2}
top(1,)      -> 15    | levels=[15] cnt={15: 2}
reset(1,)      -> None  | levels=[15] cnt={15: 1}
top(1,)      -> 15    | levels=[15] cnt={15: 1}


## 9. Code nộp LeetCode

Với ≤ 1000 lời gọi, cách gọn và an toàn nhất là `dict` + `heapq.nlargest` (chỉ dùng thư viện chuẩn).

In [15]:
# Bài thiết kế: LeetCode không dùng class Solution mà khởi tạo trực tiếp class Leaderboard.
import heapq
from collections import defaultdict


class Leaderboard:

    def __init__(self):
        self.score = defaultdict(int)

    def addScore(self, playerId: int, score: int) -> None:
        self.score[playerId] += score

    def top(self, K: int) -> int:
        return sum(heapq.nlargest(K, self.score.values()))

    def reset(self, playerId: int) -> None:
        del self.score[playerId]


# Your Leaderboard object will be instantiated and called as such:
# obj = Leaderboard()
# obj.addScore(playerId,score)
# param_2 = obj.top(K)
# obj.reset(playerId)

assert run_ops(Leaderboard, ops1, args1) == want1
print("Leaderboard OK trên ví dụ đề")

Leaderboard OK trên ví dụ đề


Nếu bạn muốn `top` nhanh hơn khi số lời gọi `top` rất lớn, có thể nộp Cách 5 với `from sortedcontainers import SortedList` (LeetCode có cài sẵn thư viện này), thay `bisect.insort` bằng `add` và `pop(bisect...)` bằng `remove`.

## 10. Vì sao đúng

Gọi `S` là ánh xạ "người chơi → điểm" theo **định nghĩa của đề** sau mỗi thao tác.

- **Invariant A (mọi cách):** `self.score` luôn bằng `S`. `addScore` cộng dồn (hoặc tạo mới với giá trị `score`, vì `0 + score = score`) đúng như đề; `reset` xoá người chơi — tương đương "điểm về 0 và rời bảng".
- **`top(K)` đúng:** tổng K giá trị lớn nhất của multiset `S.values()`. Sắp xếp giảm dần, `nlargest`, và min-heap kích thước K đều trả về đúng K giá trị lớn nhất (chuẩn của thư viện / lập luận: một giá trị bị đẩy khỏi heap luôn nhỏ hơn K giá trị còn lại).
- **Invariant B (Cách 4, 5):** multiset trong `ranked` (hoặc `cnt`) bằng multiset `S.values()`. Mỗi lần đổi điểm ta xoá **đúng một** bản của điểm cũ (`bisect_left` tìm thấy một vị trí có giá trị đó, vì invariant đảm bảo nó tồn tại) rồi chèn điểm mới. Khi đó `ranked[-K:]` là K giá trị lớn nhất.
- Trùng điểm không gây vấn đề: ta chỉ cần **tổng**, nên không quan tâm *ai* trong số những người bằng điểm được chọn.

Đảm bảo `K ≤ số người chơi` nên không cần xử lý trường hợp thiếu người.

## 11. Time / Space Complexity

Đặt `P` = số người chơi hiện tại (≤ 1000 do số lời gọi), `D` = số mức điểm khác nhau (≤ `P`), `K` = tham số của `top`.

| Cách | `addScore` | `reset` | `top(K)` | Space |
|---|---|---|---|---|
| 1. Sort mỗi lần top | O(1) | O(1) | `O(P log P)` | `O(P)` |
| 2. `nlargest` | O(1) | O(1) | `O(P log K)` | `O(P)` |
| 3. Min-heap K viết tay | O(1) | O(1) | `O(P log K)` | `O(P + K)` |
| 4. Bucket + levels | `O(D)` worst (`O(log D)` với cây cân bằng) | `O(D)` | `O(min(K, D))` | `O(P)` |
| 5. Sorted list | `O(P)` worst (`O(log P)` với `SortedList`) | `O(P)` | `O(K)` | `O(P)` |

Tất cả `O(1)` trung bình cho thao tác `dict`.

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [16]:
ALL = [LeaderboardSort, LeaderboardHeap, LeaderboardHeapK, LeaderboardBucket, LeaderboardSortedList, Leaderboard]


def check(ops, args, want=None):
    outs = [run_ops(cls, ops, args) for cls in ALL]
    assert all(o == outs[0] for o in outs), outs
    if want is not None:
        assert outs[0] == want, (outs[0], want)
    return outs[0]


check(ops1, args1, want1)
# một người chơi, cộng dồn nhiều lần
check(["Leaderboard", "addScore", "addScore", "top"], [[], [7, 100], [7, 100], [1]], [None, None, None, 200])
# reset rồi thêm lại -> bắt đầu từ 0
check(["Leaderboard", "addScore", "reset", "addScore", "top"], [[], [1, 50], [1], [1, 5], [1]], [None, None, None, None, 5])
# nhiều người trùng điểm, top lấy cắt ngang một mức điểm
check(["Leaderboard", "addScore", "addScore", "addScore", "addScore", "top", "top"],
      [[], [1, 10], [2, 10], [3, 10], [4, 1], [2], [4]], [None] * 5 + [20, 31])
# top(K) với K = số người chơi
check(["Leaderboard", "addScore", "addScore", "top"], [[], [1, 3], [2, 4], [2]], [None, None, None, 7])
print("edge cases OK")

edge cases OK


### 12.2 Ngẫu nhiên có seed: đối chiếu mọi cách

Sinh chuỗi thao tác hợp lệ: `reset` chỉ gọi với người chơi đang có mặt, `top(K)` với `1 ≤ K ≤ số người chơi`.

In [17]:
def random_ops(rng, n_calls, id_range):
    ops, args = ["Leaderboard"], [[]]
    present = set()
    for _ in range(n_calls):
        r = rng.random()
        if r < 0.5 or not present:
            p = rng.randint(1, id_range)
            ops.append("addScore")
            args.append([p, rng.randint(1, 100)])
            present.add(p)
        elif r < 0.75:
            ops.append("top")
            args.append([rng.randint(1, len(present))])
        else:
            p = rng.choice(sorted(present))
            ops.append("reset")
            args.append([p])
            present.discard(p)
    return ops, args


rng = random.Random(1244)
for _ in range(300):
    ops, args = random_ops(rng, rng.randint(1, 60), rng.choice([3, 10, 50]))
    check(ops, args)
print("300 chuỗi thao tác ngẫu nhiên OK")

300 chuỗi thao tác ngẫu nhiên OK


### 12.3 Test giới hạn: 1000 lời gọi, và một bài "stress" lớn hơn đề để thấy khác biệt

In [18]:
rng = random.Random(2024)
ops, args = random_ops(rng, 1000, 10000)
check(ops, args)

ops_big, args_big = random_ops(random.Random(5), 20000, 3000)
for cls in ALL:
    t = time.perf_counter()
    run_ops(cls, ops_big, args_big)
    print(f"{cls.__name__:22} 20000 lời gọi: {(time.perf_counter() - t) * 1000:6.1f} ms")

LeaderboardSort        20000 lời gọi:  625.2 ms


LeaderboardHeap        20000 lời gọi: 1117.9 ms


LeaderboardHeapK       20000 lời gọi:  741.4 ms
LeaderboardBucket      20000 lời gọi:  109.0 ms
LeaderboardSortedList  20000 lời gọi:   45.9 ms


Leaderboard            20000 lời gọi: 1379.3 ms


Nhận xét (số đo cụ thể tuỳ máy): trong bài stress, `K` được chọn ngẫu nhiên tới tận `P`, nên `heapq.nlargest` **không** nhanh hơn `sorted` (khi `K ≈ P`, `O(P log K) ≈ O(P log P)` và hằng số của heap lớn hơn). Hai cách giữ cấu trúc có thứ tự (bucket, sorted list) thắng rõ vì `top` chỉ đọc phần cuối. Với giới hạn thật của đề (≤ 1000 lời gọi) thì mọi cách đều chạy tức thì.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Ghi đè điểm thay vì cộng dồn

In [19]:
class LeaderboardOverwrite(LeaderboardHeap):
    def addScore(self, playerId, score):
        self.score[playerId] = score          # SAI: phải là +=


ops = ["Leaderboard", "addScore", "addScore", "top"]
args = [[], [1, 10], [1, 20], [1]]
print("sai :", run_ops(LeaderboardOverwrite, ops, args))
print("đúng:", run_ops(Leaderboard, ops, args))

sai : [None, None, None, 20]
đúng: [None, None, None, 30]


**Vì sao sai:** đề nói "cộng `score` vào điểm hiện có". Người chơi 1 phải có `30`, không phải `20`. **Sửa:** `self.score[playerId] += score` (với `defaultdict(int)`).

### Lỗi 2 — Cấu trúc sắp xếp: quên xoá điểm cũ khi cộng thêm

In [20]:
class LeaderboardStale(LeaderboardSortedList):
    def addScore(self, playerId, score):
        self.score[playerId] = self.score.get(playerId, 0) + score
        bisect.insort(self.ranked, self.score[playerId])     # SAI: điểm cũ vẫn nằm trong ranked


ops = ["Leaderboard", "addScore", "addScore", "addScore", "top"]
args = [[], [1, 10], [1, 20], [2, 5], [2]]
print("sai :", run_ops(LeaderboardStale, ops, args))
print("đúng:", run_ops(Leaderboard, ops, args))

sai : [None, None, None, None, 40]
đúng: [None, None, None, None, 35]


**Vì sao sai:** `ranked = [5, 10, 30]` chứa "bóng ma" `10` của người chơi 1 → `top(2) = 40` thay vì `30 + 5 = 35`. Invariant B bị phá. **Sửa:** xoá một bản điểm cũ trước khi chèn điểm mới (Cách 5).

### Lỗi 3 — `top` dùng heap chung rồi `pop`, làm mất dữ liệu

In [21]:
class LeaderboardDestroy:
    def __init__(self):
        self.score = defaultdict(int)

    def addScore(self, playerId, score):
        self.score[playerId] += score

    def top(self, K):
        self.h = [-v for v in self.score.values()]
        heapq.heapify(self.h)
        total = 0
        for _ in range(K):
            total -= heapq.heappop(self.h)
        return total

    def reset(self, playerId):
        del self.score[playerId]


ops = ["Leaderboard", "addScore", "addScore", "top", "top"]
args = [[], [1, 10], [2, 20], [1], [2]]
print("kết quả:", run_ops(LeaderboardDestroy, ops, args), "| đúng:", run_ops(Leaderboard, ops, args))

kết quả: [None, None, None, 20, 30] | đúng: [None, None, None, 20, 30]


Phiên bản trên tình cờ đúng vì heap được **dựng lại** mỗi lần `top`. Lỗi thật xảy ra khi người ta muốn "tối ưu" bằng cách giữ heap làm thuộc tính lâu dài và pop trực tiếp từ nó:

In [22]:
class LeaderboardPersistentHeap:
    def __init__(self):
        self.score = defaultdict(int)
        self.h = []

    def addScore(self, playerId, score):
        self.score[playerId] += score
        heapq.heappush(self.h, -self.score[playerId])     # vừa sai vì giá trị cũ còn lại...

    def top(self, K):
        return -sum(heapq.heappop(self.h) for _ in range(K))   # ...vừa phá heap

    def reset(self, playerId):
        del self.score[playerId]


ops = ["Leaderboard", "addScore", "addScore", "top", "top"]
args = [[], [1, 10], [2, 20], [1], [1]]
print("sai :", run_ops(LeaderboardPersistentHeap, ops, args))
print("đúng:", run_ops(Leaderboard, ops, args))

sai : [None, None, None, 20, 10]
đúng: [None, None, None, 20, 20]


**Vì sao sai:** `top(1)` lần đầu pop mất `20`; lần hai còn lại `10` nên trả `10` thay vì `20`. `top` là truy vấn **chỉ đọc**, không được thay đổi trạng thái. **Sửa:** tính trên bản sao (`nlargest` không sửa dữ liệu gốc) hoặc dùng cấu trúc hỗ trợ đọc K phần tử mà không xoá.

### Lỗi 4 — Sắp xếp tăng dần rồi lấy K phần tử đầu

In [23]:
class LeaderboardAscending(LeaderboardSort):
    def top(self, K):
        return sum(sorted(self.score.values())[:K])        # SAI: K nhỏ nhất


print("sai :", run_ops(LeaderboardAscending, ops1, args1))
print("đúng:", run_ops(Leaderboard, ops1, args1))

sai : [None, None, None, None, None, None, 4, None, None, None, 94]
đúng: [None, None, None, None, None, None, 73, None, None, None, 141]


**Vì sao sai:** `sorted` mặc định **tăng dần**, `[:K]` là K điểm **thấp** nhất. **Sửa:** `sorted(..., reverse=True)[:K]` hoặc `sorted(...)[-K:]`.

## 14. Mẹo & nhận dạng pattern

- Bài **Design**: liệt kê từng thao tác và tần suất của nó, rồi chọn cấu trúc dữ liệu cân bằng chi phí. Luôn bắt đầu bằng phiên bản đơn giản đúng (dict + sort) rồi mới tối ưu.
- "Tổng/top K lớn nhất" → `heapq.nlargest` hoặc min-heap kích thước K.
- Giá trị là **số nguyên bị chặn**, nhiều phần tử trùng → **đếm theo giá trị** (bucket/`Counter`).
- Cần cập nhật + truy vấn thứ hạng liên tục → **sorted list / cây cân bằng / Fenwick tree theo giá trị**.
- Mọi cấu trúc phụ phải giữ **invariant đồng bộ** với dữ liệu gốc: sửa một chỗ thì sửa mọi chỗ.
- Viết sẵn harness `run_ops` để test bài Design giống trình chấm.

## 15. Biến thể

1. **Trả về danh sách ID của top K** (phá hoà bằng ID nhỏ) → sắp xếp theo `(-score, id)`.
2. **Thứ hạng của một người chơi** (`rank(playerId)`) → sorted list: `len(ranked) - bisect_right(ranked, score) + 1`.
3. **Điểm rất lớn, số thao tác 10⁵** → `SortedList` hoặc Fenwick tree trên giá trị đã nén (coordinate compression).
4. **LeetCode 355 Design Twitter**, **LeetCode 1845 Seat Reservation Manager**, **LeetCode 2353 Design a Food Rating System**: cùng họ "dict + cấu trúc có thứ tự, giữ đồng bộ".
5. **Leaderboard theo cửa sổ thời gian** (chỉ tính điểm trong 24h gần nhất) → thêm hàng đợi sự kiện để trừ điểm hết hạn.

Demo biến thể 1 và 2:

In [24]:
class LeaderboardPlus(LeaderboardSortedList):
    def top_ids(self, K):
        return [p for p, _ in sorted(self.score.items(), key=lambda kv: (-kv[1], kv[0]))[:K]]

    def rank(self, playerId):
        s = self.score[playerId]
        return len(self.ranked) - bisect.bisect_right(self.ranked, s) + 1


lb = LeaderboardPlus()
for p, s in [(1, 73), (2, 56), (3, 39), (4, 51), (5, 56)]:
    lb.addScore(p, s)
print("top 3 id:", lb.top_ids(3))
print("rank:", {p: lb.rank(p) for p in range(1, 6)})

top 3 id: [1, 2, 5]
rank: {1: 1, 2: 2, 3: 5, 4: 4, 5: 2}


## 16. So sánh các cách

| Cách | Ý tưởng | Time (add / reset / top) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sort | dict, sort khi top | O(1) / O(1) / `O(P log P)` | `O(P)` | Ít lời gọi `top`; baseline |
| 2. `nlargest` | dict, heap K khi top | O(1) / O(1) / `O(P log K)` | `O(P)` | Mặc định, ngắn gọn (code nộp) |
| 3. Min-heap K | như 2, viết tay | O(1) / O(1) / `O(P log K)` | `O(P + K)` | Khi cần hiểu/tuỳ biến heap |
| 4. Bucket | đếm theo mức điểm | `O(D)` / `O(D)` / `O(min(K, D))` | `O(P)` | Nhiều người trùng điểm, `top` nhiều |
| 5. Sorted list | luôn sắp xếp mọi điểm | `O(P)` (`O(log P)` với SortedList) / như trên / `O(K)` | `O(P)` | `top` rất nhiều, có `SortedList` |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** chạy `["Leaderboard","addScore","addScore","addScore","top","reset","top"]` với `[[],[1,5],[2,7],[1,4],[1],[1],[1]]`. Kết quả?

**Bài 2 (tính tay):** sau `addScore(1,10), addScore(2,10), addScore(3,9), addScore(2,1)`, `top(2)` bằng bao nhiêu? Trạng thái `levels`/`cnt` của Cách 4 lúc đó?

**Bài 3 (điền TODO):** hoàn thiện `LeaderboardTODO` dùng `dict` + `sorted(...)[-K:]`.

**Bài 4 (tự viết lại):** viết lại Cách 4 (bucket) nhưng **không** dùng `levels`; `top` duyệt `range(max_score, 0, -1)` với `max_score` được cập nhật khi thêm điểm.

**Bài 5 (mini challenge):** thêm method `median()` trả về điểm trung vị (chọn phần tử thứ `(P−1)//2` trong thứ tự tăng dần) cho Cách 5.

In [25]:
class LeaderboardTODO:
    def __init__(self):
        # TODO
        raise NotImplementedError

    def addScore(self, playerId, score):
        raise NotImplementedError

    def top(self, K):
        raise NotImplementedError

    def reset(self, playerId):
        raise NotImplementedError


class LeaderboardScan:
    # TODO: score dict + cnt Counter + max_score; top duyệt từ max_score xuống
    pass


def median_score(lb):
    # TODO: lb là LeaderboardSortedList
    raise NotImplementedError


# Test gợi ý:
# assert run_ops(LeaderboardTODO, ops1, args1) == want1
# assert run_ops(LeaderboardScan, ops1, args1) == want1

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: theo dõi `dict` sau từng lệnh như mục 4.
- Mức 2: Bài 1: sau 3 lệnh add, người 1 có `9`, người 2 có `7`.
- Mức 3: Bài 2: điểm là `{1:10, 2:11, 3:9}`.

**Bài 3–4:**
- Mức 1: Bài 3: sao chép Cách 1, đổi cách lấy top.
- Mức 2: Bài 4: `max_score` chỉ tăng (có thể lớn hơn điểm thực tế cao nhất sau `reset`, vẫn đúng vì `cnt` các mức đó bằng 0).
- Mức 3: `for s in range(self.max_score, 0, -1): take = min(self.cnt[s], K) ...`.

**Bài 5:**
- Mức 1: `ranked` đã tăng dần.
- Mức 2: trung vị theo định nghĩa đề bài tập là `ranked[(P - 1) // 2]`.
- Mức 3: `P = len(lb.ranked)`.

## 19. Đáp án tham khảo

**Bài 1:** `[None, None, None, None, 9, None, 7]`. **Bài 2:** `top(2) = 11 + 10 = 21`; `levels = [9, 10, 11]`, `cnt = {9:1, 10:1, 11:1}`.

In [26]:
assert check(["Leaderboard", "addScore", "addScore", "addScore", "top", "reset", "top"],
             [[], [1, 5], [2, 7], [1, 4], [1], [1], [1]]) == [None, None, None, None, 9, None, 7]
lb = LeaderboardBucket()
for p, s in [(1, 10), (2, 10), (3, 9), (2, 1)]:
    lb.addScore(p, s)
assert lb.top(2) == 21 and lb.levels == [9, 10, 11] and dict(lb.cnt) == {9: 1, 10: 1, 11: 1}
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
class LeaderboardTODO:
    def __init__(self):
        self.score = defaultdict(int)

    def addScore(self, playerId, score):
        self.score[playerId] += score

    def top(self, K):
        return sum(sorted(self.score.values())[-K:])

    def reset(self, playerId):
        del self.score[playerId]


class LeaderboardScan:
    def __init__(self):
        self.score = {}
        self.cnt = Counter()
        self.max_score = 0

    def addScore(self, playerId, score):
        old = self.score.get(playerId, 0)
        if playerId in self.score:
            self.cnt[old] -= 1
        new = old + score
        self.score[playerId] = new
        self.cnt[new] += 1
        self.max_score = max(self.max_score, new)

    def top(self, K):
        total = 0
        for s in range(self.max_score, 0, -1):
            take = min(self.cnt[s], K)
            total += take * s
            K -= take
            if K == 0:
                break
        return total

    def reset(self, playerId):
        self.cnt[self.score.pop(playerId)] -= 1


def median_score(lb):
    return lb.ranked[(len(lb.ranked) - 1) // 2]


assert run_ops(LeaderboardTODO, ops1, args1) == want1
assert run_ops(LeaderboardScan, ops1, args1) == want1
rng = random.Random(77)
for _ in range(200):
    ops, args = random_ops(rng, rng.randint(1, 50), 20)
    base = run_ops(Leaderboard, ops, args)
    assert run_ops(LeaderboardTODO, ops, args) == base == run_ops(LeaderboardScan, ops, args)
lb = LeaderboardSortedList()
for p, s in [(1, 73), (2, 56), (3, 39), (4, 51), (5, 4)]:
    lb.addScore(p, s)
assert median_score(lb) == 51
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
class Leaderboard:
    def __init__(self):
        self.score = defaultdict(int)

    def addScore(self, playerId: int, score: int) -> None:
        self.score[playerId] += score

    def top(self, K: int) -> int:
        return sum(heapq.nlargest(K, self.score.values()))

    def reset(self, playerId: int) -> None:
        del self.score[playerId]
```

**Checklist trước khi submit:**

- [ ] Đúng tên class `Leaderboard` và tên method `addScore`, `top`, `reset` (tham số `K` viết hoa như đề).
- [ ] `addScore` **cộng dồn**, người mới bắt đầu từ 0.
- [ ] `reset` xoá người chơi (hoặc đưa về 0) — và xoá cả trong mọi cấu trúc phụ.
- [ ] `top` là thao tác **chỉ đọc**, lấy K **lớn nhất**.
- [ ] Nếu dùng cấu trúc sắp xếp: xoá **một** bản điểm cũ trước khi chèn điểm mới.
- [ ] Test bằng harness `run_ops` với chuỗi thao tác ngẫu nhiên.